# Online Retail Analysis - Data Cleaning and Preparation

The main objectives of this stage are:

- Understanding the raw dataset structure
- Handling data quality issues
- Converting data types where required
- Creating additional features for analysis
- Removing invalid records
- Preparing a clean dataset for further analysis

In [31]:
# Import required libraries for data analysis

import pandas as pd
import numpy as np


## 1. Loading the Dataset

The raw Online Retail dataset is loaded into a pandas DataFrame.

A copy of the original dataset is created to perform cleaning operations while preserving the raw dataset for reference.

In [34]:
# Load the raw online retail dataset

df = pd.read_csv("online_retail.csv")


# Create a copy for data cleaning and transformation

df_clean = df.copy()


# Display the first five rows to inspect the dataset

df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Dataset Preview

The first few records are displayed to understand the structure of the dataset and identify the type of information available before beginning the cleaning process.

## 2. Data Type Conversion

The InvoiceDate column is converted into datetime format.

This conversion allows date-based analysis by enabling the dataset to be analysed using time-related information.

In [37]:
# Convert InvoiceDate column into datetime format 
# This allows time-based analysis such as monthly revenue trends

df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])


# Check updated data types after conversion

df_clean.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

## 3. Removing Duplicate Records

Duplicate records are identified and removed to improve data quality.

Removing duplicates ensures that the same transaction is not counted multiple times during analysis.

In [39]:
# Count the number of rows before removing duplicates

before = len(df_clean)


# Remove duplicate transaction records to avoid double-counting during analysis

df_clean = df_clean.drop_duplicates()


# Count the number of rows after removing duplicates

after = len(df_clean)


# Display the impact of duplicate removal

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before-after)

Rows before: 541909
Rows after: 536641
Duplicates removed: 5268


### Result

The dataset initially contained 541,909 records.

After removing duplicate records, 536,641 records remained, with 5,268 duplicate rows removed.

Removing duplicate transactions improves data reliability by ensuring that each transaction is counted only once during further analysis.

In [41]:
# Check the number of missing values in each column

df_clean.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64

### Result

The missing value analysis shows that:

- The `Description` column contains 1,454 missing values.
- The `CustomerID` column contains 135,037 missing values.
- All other columns contain complete data.

Missing product descriptions will be removed because product information is required for product-level analysis.

CustomerID missing values will be retained because these transactions still contain valid sales information and can contribute to overall sales analysis.

In [44]:
# Remove records where product descriptions are missing because product details are required for analysis

df_clean = df_clean.dropna(subset=["Description"])


# Check the dataset size after removing incomplete product records

df_clean.shape

(535187, 8)

### Result

After removing records with missing product descriptions, the dataset contains 535,187 records and 8 columns.

Missing product descriptions were removed because complete product information is required for product-level analysis.

CustomerID missing values were retained because these transactions still contain valid sales information and contribute to overall sales analysis.

## 5. Feature Engineering

A new Revenue feature is created to calculate the total value generated from each transaction.

Revenue is calculated as:

Revenue = Quantity × UnitPrice

This feature provides a transaction-level measure of sales value and will be used for further analysis.neering
#A Revenue column was created by multiplying Quantity and UnitPrice.

In [48]:
# Create a Revenue column by calculating the total value of each transaction

df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

### Result

The Revenue feature has been successfully created.

This new feature allows transaction-level sales values to be analysed and provides a basis for measuring overall sales performance.

In [50]:
# Display the updated dataset to verify the newly created Revenue column

df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


### Result

The Revenue feature has been successfully created.

The new column calculates the total value generated from each transaction by multiplying Quantity and UnitPrice.

For example, a transaction with 6 units purchased at a unit price of 2.55 generates a revenue value of 15.30.

In [51]:
df_clean.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'Revenue'],
      dtype='object')

## 6. Handling Returns and Cancelled Transactions

Transactions with negative quantities represent product returns or cancelled orders.

These records are identified before removal to understand their impact on the dataset and ensure that sales analysis focuses on completed purchases.

In [52]:
# Identify transactions with negative quantities, which represent returns or cancelled orders

negative_sales = df_clean[df_clean["Quantity"] < 0]


# Count the number of negative quantity transactions

print("Negative transactions:", len(negative_sales))

Negative transactions: 9725


### Result

A total of 9,725 transactions with negative quantities were identified.

These records represent returns or cancelled orders and will be removed to ensure that the analysis focuses only on completed sales transactions.

In [55]:
# Display examples of negative quantity transactions for inspection

negative_sales.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom,-27.50
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom,-4.65
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,-19.80
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,-6.96
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,-6.96


In [116]:
# Display selected columns from negative transactions to understand return/cancellation records

negative_sales[[
    "InvoiceNo",
    "Description",
    "Quantity",
    "UnitPrice",
    "InvoiceDate"
]].head(10)

,InvoiceNo,Description,Quantity,UnitPrice,InvoiceDate
141,C536379,Discount,-1,27.50,2010-12-01 09:41:00
154,C536383,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,2010-12-01 09:49:00
235,C536391,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,2010-12-01 10:24:00
236,C536391,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,2010-12-01 10:24:00
237,C536391,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,2010-12-01 10:24:00
238,C536391,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,2010-12-01 10:24:00
239,C536391,CHICK GREY HOT WATER BOTTLE,-12,3.45,2010-12-01 10:24:00
240,C536391,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65,2010-12-01 10:24:00
241,C536391,PLASTERS IN TIN SKULLS,-24,1.65,2010-12-01 10:24:00
939,C536506,JAM MAKING SET WITH JARS,-6,4.25,2010-12-01 12:38:00


### Transaction Review

A sample of negative quantity transactions is displayed to verify that these records represent returns or cancelled purchases.

In [119]:
# Remove transactions with negative quantities to keep only completed sales records

df_clean = df_clean[df_clean["Quantity"] > 0]


# Check the dataset size after removing return transactions

df_clean.shape

(524878, 9)

### Result

After removing transactions with negative quantities, the dataset contains 524,878 records and 9 columns.

The remaining records represent completed purchase transactions, making the dataset more suitable for sales analysis.

## 7. Removing Invalid Prices

Transactions with zero or negative UnitPrice values do not represent valid sales.

These records are identified and removed to ensure accurate revenue calculations.

In [125]:
# Check the minimum quantity value after removing return transactions

df_clean["Quantity"].min()


0.001

In [127]:
# Identify transactions with zero or negative UnitPrice values

df_clean[df_clean["UnitPrice"] <= 0].shape




(0, 9)

In [131]:
# Display examples of transactions with invalid prices

df_clean[df_clean["UnitPrice"] <= 0].head()




0.001

In [75]:
# Check the minimum UnitPrice value before removing invalid prices

df_clean["UnitPrice"].min()

-11062.06

### Result

No transactions with zero or negative UnitPrice values were found in the dataset.

The minimum UnitPrice value is 0.001, confirming that all remaining transactions contain valid positive prices.

In [78]:
# Remove transactions with zero or negative UnitPrice values

df_clean = df_clean[df_clean["UnitPrice"] > 0]



(524878, 9)

In [134]:

# Check the dataset size after price validation

df_clean.shape

(524878, 9)

### Result

After validating UnitPrice values, the dataset size remains unchanged at 524,878 records and 9 columns.

No additional records were removed because all remaining transactions contained valid positive prices.

## 8. Final Data Quality Validation

Final checks are performed to confirm that the cleaned dataset has no remaining data quality issues.

The validation includes:
- Checking for missing values
- Checking for duplicate records
- Confirming data types

In [140]:




# Verify the final data types of all columns

df_clean.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
Revenue               float64
dtype: object

In [142]:
# Check for any remaining missing values in the cleaned dataset

df_clean.isnull().sum()


InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132186
Country             0
Revenue             0
dtype: int64

In [144]:
# Check whether any duplicate records remain

df_clean.duplicated().sum()


0

### Result

The final data quality checks confirm that:

- No missing values remain in the required transaction and product-related fields.
- CustomerID contains 132,186 missing values; these records were retained because they still contain valid transaction information.
- No duplicate records remain in the cleaned dataset.
- All columns have the expected data types after the cleaning process.

The dataset is now prepared and ready for further analysis.

## 9. Exporting Clean Dataset

The cleaned dataset is exported as a CSV file for future use.



In [94]:
# Create the output directory if it does not already exist

import os

os.makedirs("data/cleaned", exist_ok=True)


# Export the cleaned dataset as a CSV file

df_clean.to_csv(
    "data/cleaned/online_retail_cleaned.csv",
    index=False
)


# Confirm that the file has been successfully created

os.path.exists("data/cleaned/online_retail_cleaned.csv")

### Final Conclusion

The data cleaning process has been completed successfully.

The dataset was prepared by:
- Removing duplicate records
- Handling missing product information
- Creating a Revenue feature
- Removing return/cancelled transactions
- Validating transaction prices
- Performing final quality checks

The cleaned dataset is now ready for further analysis.